In [ ]:
from plotlywaterfall import Waterfall
import pandas as pd

# simple white background
import plotly.io as pio
pio.templates["custom"] = pio.templates["plotly_white"]
pio.templates["custom"].layout.autosize = False
pio.templates["custom"].layout.width = 1200
pio.templates["custom"].layout.height = 600
pio.templates.default = "custom"

# Simple Examples

Most simple plot: simply having some `X` and `Y` values to plot. Note that values with same `X` are summed. Without categories or groups, bars are coloured by direction: increases, decreases and totals:

In [ ]:
df = pd.DataFrame({
    "X": ["Product Sales", "Product Sales", "Services Sales", "Product R&D", "Product R&D", "Services COGS"],
    "Y": [3, 4, 1, -1, -2, -0.5]
})
fig = Waterfall(df, x="X", y="Y").get_fig()
fig.update_yaxes(title="Cashflow [MUSD]")
fig

Now let's add the total sum:

In [ ]:
df = pd.DataFrame({
    "X": ["Product Sales", "Product Sales", "Services Sales", "Product R&D", "Product R&D", "Services COGS"],
    "Y": [3, 4, 1, -1, -2, -0.5]
})
fig = Waterfall(df, x="X", y="Y", total=True).get_fig()
fig.update_yaxes(title="Cashflow [MUSD]")
fig

More useful is to add the category:

In [ ]:
df = pd.DataFrame({
    "X": ["Product Sales", "Product Sales", "Services Sales", "Product R&D", "Product R&D", "Services COGS"],
    "Y": [3, 4, 1, -1, -2, -0.5],
    "category": ["Product A", "Product B", "Plugins", "Product A", "Product B", "Plugins"]
})
fig = Waterfall(df, x="X", y="Y", category="category", total=True).get_fig()
fig.update_yaxes(title="Cashflow [MUSD]")
fig

The total can also be negative. Here *Product R&D* is negative for both products:

In [ ]:
df = pd.DataFrame({
    "X": ["Product Sales", "Product Sales", "Licenses", "Product R&D", "Product R&D", "License Mgmt"],
    "Y": [3, 4, 1, -4, -5, -2],
    "category": ["Product A", "Product B", "Plugins", "Product A", "Product B", "Plugins"]
})
fig = Waterfall(df, x="X", y="Y", category="category", total=True).get_fig()
fig.update_yaxes(title="Cashflow [MUSD]")
fig

## Mixed signs

Categories of one bar can also have different signs, e.g. a subsidy (+1) and a restructuring charge (-4) at the same `X`, or a total combining positive and negative categories. Such a bar is split into one narrow sub-step per category, walking from the level where the bar starts to where it ends. A light *net ghost* bar behind the sub-steps shows the net change:

In [ ]:
df = pd.DataFrame({
    "X": ["Revenue", "Costs", "Other", "Other"],
    "Y": [9, -7, 1, -4],
    "category": ["Operating", "Operating", "Operating", "One-off"]
})
Waterfall(df, x="X", y="Y", category="category", total=True).get_fig()

The order of the sub-steps follows the data by default; `step_order="positive_first"` or `"largest_first"` changes it. `net_ghost=False` hides the net bar (or pass a dict of plotly marker properties to style it), and `mixed_signs="error"` restores the old behaviour of raising an exception:

In [ ]:
Waterfall(df, x="X", y="Y", category="category", total=True, step_order="largest_first", net_ghost=False).get_fig()

## Grouping

One can also can create multiple groups. Grouping might be useful to compare data. Let's use in the following this dataframe: 

In [ ]:
df = pd.DataFrame({
    "X": ["A", "B", "C"]*4 + ["D", "D" ], 
    "Y": [4, 1, 8, 7, 3, 2] + [i-1 for i in [4, 1, 8, 7, 3, 2]] + [8, 5],
    "category": ["one"]*3+["two"]*3 + ["one"]*3+["two"]*3 + ["two", "three"],
    "group": ["Group1"]*6 + ["Group2"]*6 + ["Group2", "Group3"]
})
df

In [ ]:
c = Waterfall(df, x="X", y="Y", category="category", group="group", total=True)
fig = c.get_fig()
fig

## Coloring

The `color` argument says what drives the colours: the category column (default if there is one), the group column, or `"direction"` (increase / decrease / total; default for charts without category and group). Colours are assigned per value, so a category has the same colour in every group.

`color_discrete_map` sets colours per value; values not in the map get colours from `color_discrete_sequence` (default: the colorway of the plotly template). `pattern_shape` adds fill patterns as a second visual cue, e.g. colour by category and pattern by group:

In [ ]:
c = Waterfall(
    df, x="X", y="Y", category="category", group="group", total=True,
    color_discrete_map={"one": "#2a78d6", "two": "#eb6834", "three": "#1baf7a"},
    pattern_shape="group",
)
c.get_fig()

Colour by group instead:

In [ ]:
c = Waterfall(
    df, x="X", y="Y", category="category", group="group", total=True,
    color="group",
    color_discrete_map={"Group1": "steelblue", "Group2": "darkorange", "Group3": "seagreen"},
)
c.get_fig()

For full control, keys can also be `(group, category)` tuples. They take precedence over plain values:

In [ ]:
colors = {
    "one": "red", "two": "blue", "three": "green",
    ("Group2", "one"): "salmon", ("Group2", "two"): "lightskyblue",
}
c = Waterfall(df, x="X", y="Y", category="category", group="group", total=True, color_discrete_map=colors)
c.get_fig()

Direction colours also work with categories and groups. Combine them with a pattern so that the categories stay distinguishable; `color_discrete_map` keys `"increase"`, `"decrease"` and `"total"` override the default green / red / grey:

In [ ]:
c = Waterfall(
    df, x="X", y="Y", category="category", group="group", total=True,
    color="direction", pattern_shape="category",
)
c.get_fig()

Keys of `color_discrete_map` that match no value raise a warning, so typos do not go unnoticed.

The older `colors` argument still works but is deprecated and will be removed in version 1.0:

| Old `colors=` | New |
|---|---|
| `{"one": "red", "two": "blue"}` (per category) | `color_discrete_map={"one": "red", "two": "blue"}` |
| `{"Group1": "red", "Group2": "blue"}` (per group) | `color="group", color_discrete_map={"Group1": "red", "Group2": "blue"}` |
| `{"Group1": {"one": "red"}}` (per group and category) | `color_discrete_map={("Group1", "one"): "red"}` |

# Subtotals

It is possible to define where subtotals should be calculated and added:

In [ ]:
c = Waterfall(df, x="X", y="Y", category="category", group="group", total=True, subtotals={"C": "Subtotal"})
c.get_fig()